# Aula 08 — Reordenação (*Rerank*): *Retrieve & Rerank* e Proximidade Ordenada (Teoria)
**Projeto Integrador III — Motor de Busca**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*


In [1]:
# 1. Setup oficial do motor08 e montagem dos 8 documentos do curso
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")
docs <- docs_aula()          # os 8 documentos do curso
cfg  <- cfg_aula()           # as decisoes canonicas (agora com modelo, lsa_k e rrf_c)
ix   <- montar(docs, cfg)    # tudo que as aulas anteriores calcularam
cat("Versao do motor:", MOTOR_VERSAO, "\n")
estado()


Versao do motor: motor08 v2 2026-10-22 
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | manter |
| `cfg$stopwords` | character | 0 |  |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `cfg$modelo` | character | 1 | bm25 |
| `cfg$lsa_k` | numeric | 1 | 2 |
| `cfg$rrf_c` | numeric | 1 | 60 |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao o..., d2 o modelo de espaco vetorial..., ... |
| `ix$tokens` | list | 8 | elementos: d1, d2, d3, d4, d5, d6, d7, d8 |


## Módulo 1 — Dois estágios: recuperar muitos, reordenar poucos
O estágio 1 (BM25, LSA ou híbrido) dá escore a todos os $N$ documentos buscando *recall* no top-$k$; o estágio 2 relê apenas os $k$ primeiros candidatos para melhorar a precisão no topo.


In [2]:
rb <- ranking_bm25("modelo de recuperacao", ix, cfg)   # estagio 1: um escore para CADA documento
round(rb, 3)                                           # o ranking da Aula 04
r <- relevancia(rb, gabarito_aula(), 2)                # relevante (grau >= 2) em cada posicao
r$rel                                                  # 1 = relevante; 0 = nao
sapply(1:4, function(k) recall_k(r$rel, k, r$R))       # que fracao dos relevantes ja esta no top-k

# Checkpoint 1: limiar 1 (d1, d2, d3, d6)
r1 <- relevancia(rb, gabarito_aula(), 1)
cat("Recall@k com limiar 1:\n")
sapply(1:8, function(k) recall_k(r1$rel, k, r1$R))


   d3    d1    d2    d4    d8    d6    d5    d7 
1.873 1.869 1.687 1.427 0.519 0.492 0.000 0.000 
[1] 1 0 1 0 0 0 0 0
[1] 0.5 0.5 1.0 1.0
Recall@k com limiar 1:
[1] 0.25 0.50 0.75 0.75 0.75 1.00 1.00 1.00


## Módulo 2 — Pontuar cada lado separado × ler o par
Ao inverter a ordem das palavras de `d2`, o BM25 permanece idêntico (`1.687`), pois é um modelo de saco de palavras (*bag of words*).


In [3]:
docs2 <- docs                                                # uma copia do corpus
docs2[["d2"]] <- paste(rev(ix$tokens$d2), collapse = " ")    # d2 de tras para frente
docs2[["d2"]]                                                # confira: as mesmas palavras, outra ordem
ix2 <- montar(docs2, cfg)                                    # o indice do corpus embaralhado
round(ranking_bm25("modelo de recuperacao", ix2, cfg), 3)    # mudou alguma coisa?


[1] "vetores como documentos representa vetorial espaco de modelo o"
   d3    d1    d2    d4    d8    d6    d5    d7 
1.873 1.869 1.687 1.427 0.519 0.492 0.000 0.000 


## Módulo 3 — Um reordenador que lê os dois juntos: a proximidade ordenada
Implementação de `proximidade(qt, dt, janela = 3)` com `outer()`, `min()`, `&&` e laços aninhados.


In [4]:
proximidade <- function(qt, dt, janela = 3) {        # qt: tokens da consulta; dt: tokens do documento
  s <- 0                                             # o escore comeca em zero
  for (i in seq_along(qt)) for (j in seq_along(qt))  # todos os pares de posicoes da consulta...
    if (i < j && qt[i] != qt[j]) {                   # ...com i antes de j e termos diferentes
      pos_i <- which(dt == qt[i])                    # onde o termo i aparece no documento
      pos_j <- which(dt == qt[j])                    # onde o termo j aparece
      if (length(pos_i) && length(pos_j)) {          # os dois aparecem?
        d <- outer(pos_j, pos_i, "-")                # todas as distancias (j menos i)
        d <- d[d > 0 & d <= janela]                  # so j DEPOIS de i, e dentro da janela
        if (length(d)) s <- s + 1 / min(d)           # soma 1 / a menor distancia
      }
    }
  s                                                  # devolve o escore
}

qt <- preparar("modelo de recuperacao", cfg)                 # modelo, de, recuperacao
px <- sapply(ix$tokens, function(dt) proximidade(qt, dt))    # um escore por documento
px                                                           # quem tem os termos em ordem e perto?
proximidade(qt, ix2$tokens$d2)                               # e o d2 embaralhado do Modulo 2?
cat("Checkpoint 3 ('modelo de texto' em d3 e d2):",
    proximidade(preparar("modelo de texto", cfg), ix$tokens$d3), "vs",
    proximidade(preparar("modelo de texto", cfg), ix$tokens$d2), "\n")


 d1  d2  d3  d4  d5  d6  d7  d8 
0.0 1.0 0.5 0.0 0.0 0.0 0.0 0.0 
[1] 0
Checkpoint 3 ('modelo de texto' em d3 e d2): 1.5 vs 1 


## Módulo 4 — Reordenar o top-$k$ e medir: o estágio 1 limita o estágio 2
Implementação de `reordenar(rk, k, sc)` mantendo a ordem do estágio 1 em caso de empate e preservando a cauda `names(rk)[-(1:k)]`.


In [5]:
reordenar <- function(rk, k, sc) {               # rk: ranking do estagio 1; sc: escores do estagio 2
  k <- min(k, length(rk))                        # protecao: k nao passa do numero de documentos
  top <- names(rk)[1:k]                          # a cabeca: os k primeiros
  nova <- top[order(-sc[top], seq_along(top))]   # pelo escore, decrescente; empate: ordem do estagio 1
  c(nova, names(rk)[-(1:k)])                     # cabeca reordenada + cauda intacta (NOMES)
}
reordenar(rb, 2, px)            # so d3 e d1 sao relidos
reordenar(rb, 3, px)            # agora d2 entra na cabeca

g <- gabarito_aula()                                       # os graus canonicos
round(rbind(bm25 = metricas(rb, g, 2),                     # o estagio 1 sozinho
            k2   = metricas(reordenar(rb, 2, px), g, 2),   # reordenando 2
            k3   = metricas(reordenar(rb, 3, px), g, 2)), 3)  # reordenando 3


[1] "d3" "d1" "d2" "d4" "d8" "d6" "d5" "d7"
[1] "d2" "d3" "d1" "d4" "d8" "d6" "d5" "d7"
     P_at_k    AP RR nDCG_bin nDCG_grad
bm25  0.667 0.833  1     0.92     0.951
k2    0.667 0.833  1     0.92     0.951
k3    0.667 1.000  1     1.00     0.982


## Módulos 5 e 6 — Os limites do brinquedo e a curva Latência × Qualidade ($k$ como hiperparâmetro)
Verificação da sensibilidade à ordem literal (`"documentos busca"`) e medição do $\text{AP}$ para $k = 1..8$.


In [6]:
qb <- preparar("documentos busca", cfg)                       # a ordem trocada
round(ranking_bm25("documentos busca", ix, cfg), 3)           # o BM25 nao liga para ordem
sapply(ix$tokens, function(dt) proximidade(qb, dt))           # o brinquedo liga - demais

cat("AP por k (1 a 8):\n")
round(sapply(1:8, function(k) metricas(reordenar(rb, k, px), g, 2)[["AP"]]), 3)


   d5    d7    d1    d6    d2    d3    d4    d8 
1.878 1.219 0.731 0.693 0.659 0.000 0.000 0.000 
d1 d2 d3 d4 d5 d6 d7 d8 
 0  0  0  0  0  0  0  0 
AP por k (1 a 8):
[1] 0.833 0.833 1.000 1.000 1.000 1.000 1.000 1.000
